In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

counties["features"][0]

{'type': 'Feature',
 'properties': {'GEO_ID': '0500000US01001',
  'STATE': '01',
  'COUNTY': '001',
  'NAME': 'Autauga',
  'LSAD': 'County',
  'CENSUSAREA': 594.436},
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-86.496774, 32.344437],
    [-86.717897, 32.402814],
    [-86.814912, 32.340803],
    [-86.890581, 32.502974],
    [-86.917595, 32.664169],
    [-86.71339, 32.661732],
    [-86.714219, 32.705694],
    [-86.413116, 32.707386],
    [-86.411172, 32.409937],
    [-86.496774, 32.344437]]]},
 'id': '01001'}

In [3]:
import plotly.express as px

df = px.data.gapminder().query("year==2007")
fig = px.choropleth(df, locations="iso_alpha",
                    color="lifeExp", # lifeExp is a column of gapminder
                    hover_name="country", # column to add to hover information
                    color_continuous_scale=px.colors.sequential.Plasma)
fig.show()

ModuleNotFoundError: No module named 'plotly'

In [4]:
import pandas as pd
import numpy as np

## COVID-19

In [5]:
url = "https://covid19.who.int/WHO-COVID-19-global-data.csv"
url_iso = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"

In [6]:
data = pd.read_csv(
    url,
    sep=',',
    encoding='utf-8',
)

data_iso = pd.read_csv(
    url_iso,
    sep=',',
    encoding='utf-8',
)

HTTPError: HTTP Error 403: Forbidden

In [ ]:
data = data.drop(['Country', 'WHO_region', 'New_cases', 'New_deaths',
       'Cumulative_deaths'],axis=1)

In [ ]:
data_iso = data_iso.drop(['UID','FIPS', 'Admin2', 'Province_State', 'Lat', 'Long_', 'Combined_Key', 'Population',"code3"],axis=1)

In [ ]:
maindata = pd.merge(data,data_iso,left_on="Country_code",right_on="iso2")

In [ ]:
maindata["Cumulative_cases"] = np.where(maindata["Cumulative_cases"] > 0, 1, 0)

In [ ]:
import plotly.express as px

fig = px.choropleth(maindata, locations="iso3",
                    color="Cumulative_cases", # lifeExp is a column of gapminder
                    hover_name="iso3", # column to add to hover information
                    animation_frame="Date_reported",
                    color_continuous_scale=px.colors.sequential.Plasma)
fig.show()

## Monkeypox

In [4]:
url = "https://raw.githubusercontent.com/owid/monkeypox/main/owid-monkeypox-data.csv"

In [19]:
data = pd.read_csv(
    url,
    sep=',',
    encoding='utf-8',
)

In [30]:
data

,location,iso_code,date,total_cases,total_deaths,new_cases,new_deaths,new_cases_smoothed,new_deaths_smoothed,new_cases_per_million,total_cases_per_million,new_cases_smoothed_per_million,new_deaths_per_million,total_deaths_per_million,new_deaths_smoothed_per_million
0,Africa,OWID_AFR,2022-05-01,27.0,2.0,0.0,0.0,0.29,0.00,0.000,0.019,0.000,0.0,0.00140,0.00000
1,Africa,OWID_AFR,2022-05-02,27.0,2.0,0.0,0.0,0.29,0.00,0.000,0.019,0.000,0.0,0.00140,0.00000
2,Africa,OWID_AFR,2022-05-03,27.0,2.0,0.0,0.0,0.29,0.00,0.000,0.019,0.000,0.0,0.00140,0.00000
3,Africa,OWID_AFR,2022-05-04,27.0,2.0,0.0,0.0,0.29,0.00,0.000,0.019,0.000,0.0,0.00140,0.00000
4,Africa,OWID_AFR,2022-05-05,27.0,2.0,0.0,0.0,0.29,0.00,0.000,0.019,0.000,0.0,0.00140,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54323,World,OWID_WRL,2023-11-27,92085.0,171.0,0.0,0.0,15.43,0.43,0.000,11.547,0.002,0.0,0.02144,0.00005
54324,World,OWID_WRL,2023-11-28,92179.0,171.0,94.0,0.0,16.14,0.14,0.012,11.558,0.002,0.0,0.02144,0.00002
54325,World,OWID_WRL,2023-11-29,92179.0,171.0,0.0,0.0,14.14,0.00,0.000,11.558,0.002,0.0,0.02144,0.00000
54326,World,OWID_WRL,2023-11-30,92181.0,171.0,2.0,0.0,14.14,0.00,0.000,11.559,0.002,0.0,0.02144,0.00000


In [1]:
def monkeypoxDataEnhancement(data):
    # Drop useless columns
    maindata = data.drop(['iso_code','total_deaths',
      'new_cases', 'new_deaths', 'new_cases_smoothed', 'new_deaths_smoothed',
      'new_cases_per_million', 'total_cases_per_million',
      'new_cases_smoothed_per_million', 'new_deaths_per_million',
      'total_deaths_per_million', 'new_deaths_smoothed_per_million'],axis=1)
    #  Columns
    all_dates = maindata["date"].unique().tolist()
    all_dates.sort()
    date_list = []
    total_cases = []
    iso_code = []
    dict_countries = dict(zip(maindata["location"].unique().tolist(), [0]*len(maindata["location"])))
    # Go through every date
    for date in all_dates:
      # Get slice of matrix
      slice = data[data["date"]==date]
      # Get lines of slice
      for index, row in slice.iterrows():
        dict_countries[row['location']] = row['total_cases']
      # Add values to lists
      for key in dict_countries.keys():
        date_list.append(date)
        iso_code.append(key)
        total_cases.append(dict_countries[key])
    # Create pandas dataframe
    d = {'iso_code':iso_code,'date':date_list, 'total_cases':total_cases}
    df = pd.DataFrame(data=d)
    # Get all isocode
    return df

In [47]:
maindata = monkeypoxDataEnhancement(data)

In [48]:
maindata["binary_cases"] = np.where(maindata["total_cases"] > 0, 1, 0)

In [49]:
import plotly.express as px

fig = px.choropleth(maindata, locations="iso_code",
                    color="binary_cases", # lifeExp is a column of gapminder
                    hover_name="iso_code", # column to add to hover information
                    animation_frame="date",
                    color_continuous_scale=px.colors.sequential.Plasma)
fig.show()